# Lecture 10: Householder Triangularization
## Trefethen & Bau — Numerical Linear Algebra

Python demonstrations using NumPy and Matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Householder Reflectors in 2D

$H = I - 2vv^T$ reflects across the hyperplane perpendicular to $v$.

In [ ]:
def householder(v):
    """Build Householder reflector H = I - 2 v v^T (v must be unit)."""
    v = v.reshape(-1, 1)
    return np.eye(len(v)) - 2 * v @ v.T

# Example 1: v = (0,1) -> reflect across x-axis
v1 = np.array([0.0, 1.0])
H1 = householder(v1)
print("v = (0,1): H =")
print(H1)
print(f"H @ (3,4) = {H1 @ np.array([3, 4])}\n")

# Example 2: v = (1,1)/sqrt(2) -> reflect across y = -x
v2 = np.array([1.0, 1.0]) / np.sqrt(2)
H2 = householder(v2)
print("v = (1,1)/sqrt(2): H =")
print(H2.round(10))
print(f"H @ (3,4) = {(H2 @ np.array([3, 4])).round(10)}")

In [ ]:
# Visualize 2D reflections
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

t = np.linspace(0, 2 * np.pi, 200)
circle = np.array([np.cos(t), np.sin(t)])

for ax, v, title in zip(axes,
    [np.array([0.0, 1.0]), np.array([1.0, 1.0]) / np.sqrt(2)],
    ['$v = (0,1)$: reflect across $x$-axis',
     '$v = (1,1)/\\sqrt{2}$: reflect across $y=-x$']):

    H = householder(v)
    reflected = H @ circle

    ax.scatter(circle[0], circle[1], c=t, cmap='hsv', s=3, alpha=0.5)
    ax.scatter(reflected[0], reflected[1], c=t, cmap='hsv', s=3, marker='x')

    # Show mirror line (perpendicular to v)
    perp = np.array([-v[1], v[0]])
    ax.plot([-2*perp[0], 2*perp[0]], [-2*perp[1], 2*perp[1]], 'k--', alpha=0.4, label='mirror')
    ax.annotate('', xy=0.7*v, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='red', lw=2))
    ax.text(0.7*v[0]+0.1, 0.7*v[1]+0.1, '$v$', fontsize=12, color='red')

    ax.set_aspect('equal')
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.grid(True, alpha=0.3)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle('Householder reflections in 2D (dots → crosses)', fontsize=13)
plt.tight_layout()
plt.show()

## The Trick: Reflect $x$ onto $\|x\| e_1$

Given $x = (3, 4)^T$, find $H$ such that $Hx = (5, 0)^T$.

In [ ]:
x = np.array([3.0, 4.0])
alpha = np.linalg.norm(x)
e1 = np.array([1.0, 0.0])

# Choose sign to avoid cancellation
w = x - alpha * e1
v = w / np.linalg.norm(w)

H = householder(v)
print(f"x = {x}, ||x|| = {alpha}")
print(f"v = {v.round(6)}")
print(f"H =\n{H.round(6)}")
print(f"Hx = {(H @ x).round(10)}")
print(f"H^T H = I: {np.allclose(H.T @ H, np.eye(2))}")
print(f"H^2 = I:   {np.allclose(H @ H, np.eye(2))}")

In [ ]:
# Visualize: x reflected onto the x-axis
plt.figure(figsize=(6, 6))

Hx = H @ x
plt.annotate('', xy=x, xytext=(0, 0),
             arrowprops=dict(arrowstyle='->', color='blue', lw=2))
plt.text(x[0]+0.1, x[1]+0.2, '$x = (3,4)$', fontsize=12, color='blue')

plt.annotate('', xy=Hx, xytext=(0, 0),
             arrowprops=dict(arrowstyle='->', color='green', lw=2))
plt.text(Hx[0]+0.1, Hx[1]-0.5, '$Hx = (5,0)$', fontsize=12, color='green')

# Mirror line
perp = np.array([-v[1], v[0]])
plt.plot([-4*perp[0], 4*perp[0]], [-4*perp[1], 4*perp[1]],
         'k--', alpha=0.4, label='mirror hyperplane')
plt.annotate('', xy=2*v, xytext=(0, 0),
             arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
plt.text(2*v[0]-0.8, 2*v[1], '$v$', fontsize=12, color='red')

theta = np.linspace(0, 2*np.pi, 200)
plt.plot(5*np.cos(theta), 5*np.sin(theta), 'gray', alpha=0.2)

plt.xlim(-2, 6)
plt.ylim(-3, 5)
plt.gca().set_aspect('equal')
plt.grid(True, alpha=0.3)
plt.title('Householder reflection: $x \\to \\|x\\| e_1$', fontsize=12)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

## Householder QR: Implementation

In [ ]:
def householder_qr(A):
    """Householder QR factorization. Returns Q (full) and R."""
    m, n = A.shape
    R = A.astype(float).copy()
    Q = np.eye(m)
    vs = []  # store Householder vectors

    for k in range(n):
        x = R[k:, k]
        e1 = np.zeros_like(x)
        e1[0] = 1.0
        # Choose sign to avoid cancellation
        alpha = np.sign(x[0]) * np.linalg.norm(x)
        v = x + alpha * e1
        v = v / np.linalg.norm(v)
        vs.append(v)

        # Apply reflection to R (only rows k:m)
        R[k:, k:] -= 2 * np.outer(v, v @ R[k:, k:])

        # Accumulate Q
        Q[k:, :] -= 2 * np.outer(v, v @ Q[k:, :])

    return Q.T, R, vs  # Q.T because we accumulated Q^T

# Test on the lecture's 3x2 example
A = np.array([[1, 1],
              [0, 1],
              [1, 0]], dtype=float)

Q, R, vs = householder_qr(A)
print(f"Q =\n{Q.round(6)}\n")
print(f"R =\n{R.round(6)}\n")
print(f"||QR - A|| = {np.linalg.norm(Q @ R - A):.2e}")
print(f"||Q^TQ - I|| = {np.linalg.norm(Q.T @ Q - np.eye(3)):.2e}")

### Step-by-Step Walkthrough

In [ ]:
A = np.array([[1, 1],
              [0, 1],
              [1, 0]], dtype=float)

# Step 1: zero out column 1 below diagonal
x = A[:, 0]
alpha = np.sign(x[0]) * np.linalg.norm(x)
e1 = np.array([1, 0, 0], dtype=float)
v1 = x + alpha * e1
v1 = v1 / np.linalg.norm(v1)
H1 = np.eye(3) - 2 * np.outer(v1, v1)

print("Step 1:")
print(f"  x = {x}, ||x|| = {np.linalg.norm(x):.6f}")
print(f"  v1 = {v1.round(6)}")
A1 = H1 @ A
print(f"  H1 @ A =\n{A1.round(6)}\n")

# Step 2: zero out column 2 below diagonal (only rows 1:)
x2 = A1[1:, 1]
alpha2 = np.sign(x2[0]) * np.linalg.norm(x2)
e1_2 = np.array([1, 0], dtype=float)
v2 = x2 + alpha2 * e1_2
v2 = v2 / np.linalg.norm(v2)
H2_small = np.eye(2) - 2 * np.outer(v2, v2)
H2 = np.eye(3)
H2[1:, 1:] = H2_small

print("Step 2:")
print(f"  x' = {x2.round(6)}")
print(f"  v2 = {v2.round(6)}")
R = H2 @ A1
print(f"  H2 @ H1 @ A = R =\n{R.round(6)}\n")

Q = H1 @ H2
print(f"Q = H1 @ H2 =\n{Q.round(6)}\n")
print(f"||QR - A|| = {np.linalg.norm(Q @ R - A):.2e}")

## Larger Example: Zeroing Out Column by Column

In [ ]:
np.random.seed(0)
A = np.random.randn(6, 3)

Q, R, vs = householder_qr(A)

print(f"A ({A.shape[0]}x{A.shape[1]}):")
print(A.round(4))
print(f"\nR (upper triangular):")
print(R.round(4))
print(f"\n||QR - A||    = {np.linalg.norm(Q @ R - A):.2e}")
print(f"||Q^TQ - I||  = {np.linalg.norm(Q.T @ Q - np.eye(6)):.2e}")

# Compare with NumPy
Q_np, R_np = np.linalg.qr(A, mode='complete')
print(f"\nNumPy: ||QR - A|| = {np.linalg.norm(Q_np @ R_np - A):.2e}")

## Why Householder Beats Gram-Schmidt: Stability Comparison

In [ ]:
def classical_gs(A):
    m, n = A.shape
    Q = np.zeros((m, n))
    R = np.zeros((n, n))
    for j in range(n):
        v = A[:, j].copy()
        for i in range(j):
            R[i, j] = Q[:, i] @ A[:, j]
            v -= R[i, j] * Q[:, i]
        R[j, j] = np.linalg.norm(v)
        Q[:, j] = v / R[j, j]
    return Q, R

def modified_gs(A):
    A = A.copy()
    m, n = A.shape
    Q = np.zeros((m, n))
    R = np.zeros((n, n))
    for j in range(n):
        R[j, j] = np.linalg.norm(A[:, j])
        Q[:, j] = A[:, j] / R[j, j]
        for k in range(j + 1, n):
            R[j, k] = Q[:, j] @ A[:, k]
            A[:, k] -= R[j, k] * Q[:, j]
    return Q, R

In [ ]:
np.random.seed(42)
m, n = 100, 50
kappas = np.logspace(1, 14, 20)

orth_cgs = []
orth_mgs = []
orth_hh = []

for kappa in kappas:
    U, _ = np.linalg.qr(np.random.randn(m, n))
    V, _ = np.linalg.qr(np.random.randn(n, n))
    s = np.logspace(0, -np.log10(kappa), n)
    A = U @ np.diag(s) @ V.T

    Q_c, _ = classical_gs(A)
    Q_m, _ = modified_gs(A)
    Q_h, _ = np.linalg.qr(A)  # Householder

    orth_cgs.append(np.linalg.norm(Q_c.T @ Q_c - np.eye(n)))
    orth_mgs.append(np.linalg.norm(Q_m.T @ Q_m - np.eye(n)))
    orth_hh.append(np.linalg.norm(Q_h.T @ Q_h - np.eye(n)))

eps = np.finfo(float).eps

plt.figure(figsize=(9, 5))
plt.loglog(kappas, orth_cgs, 'ro-', markersize=4, label='Classical GS')
plt.loglog(kappas, orth_mgs, 'bs-', markersize=4, label='Modified GS')
plt.loglog(kappas, orth_hh, 'g^-', markersize=4, label='Householder (NumPy)')
plt.loglog(kappas, kappas * eps, 'k--', alpha=0.5, label=r'$\kappa \cdot \epsilon$')
plt.axhline(1, color='gray', linestyle=':', alpha=0.5, label='Total loss')
plt.xlabel(r'$\kappa(A)$')
plt.ylabel(r'$\|Q^TQ - I\|$')
plt.title('Loss of orthogonality: CGS vs MGS vs Householder')
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## Implicit $Q$: Applying Reflectors Without Forming $Q$

In [ ]:
np.random.seed(5)
m, n = 8, 4
A = np.random.randn(m, n)
b = np.random.randn(m)

Q, R, vs = householder_qr(A)

# Method 1: Form Q explicitly, compute Q^T b
Qtb_explicit = Q.T @ b

# Method 2: Apply reflectors implicitly (the efficient way)
z = b.copy()
for k, v in enumerate(vs):
    z[k:] -= 2 * v * (v @ z[k:])
Qtb_implicit = z

print(f"Q^T b (explicit): {Qtb_explicit[:n].round(10)}")
print(f"Q^T b (implicit): {Qtb_implicit[:n].round(10)}")
print(f"Difference:       {np.linalg.norm(Qtb_explicit - Qtb_implicit):.2e}")

# Solve least squares using the implicit approach
R_thin = R[:n, :n]
x_ls = np.linalg.solve(R_thin, Qtb_implicit[:n])
x_np = np.linalg.lstsq(A, b, rcond=None)[0]
print(f"\nLeast squares solution (implicit): {x_ls.round(10)}")
print(f"Least squares solution (NumPy):    {x_np.round(10)}")
print(f"Difference: {np.linalg.norm(x_ls - x_np):.2e}")

## Storage Savings: Householder Vectors vs. Full $Q$

In [ ]:
print(f"{'m':>6} {'n':>6} {'Q entries':>12} {'HH vectors':>12} {'Savings':>10}")
print('-' * 50)
for m, n in [(100, 10), (1000, 50), (10000, 100), (100000, 200)]:
    q_entries = m * m
    hh_entries = m * n - n * (n - 1) // 2
    print(f"{m:6d} {n:6d} {q_entries:12,d} {hh_entries:12,d} {q_entries/hh_entries:9.0f}x")

## Visualizing Householder QR: Column-by-Column Elimination

In [ ]:
np.random.seed(2)
A = np.random.randn(5, 3)

R_steps = [A.copy()]
R_work = A.astype(float).copy()

for k in range(3):
    x = R_work[k:, k]
    e1 = np.zeros_like(x)
    e1[0] = 1.0
    alpha = np.sign(x[0]) * np.linalg.norm(x)
    v = x + alpha * e1
    v = v / np.linalg.norm(v)
    R_work[k:, k:] -= 2 * np.outer(v, v @ R_work[k:, k:])
    R_steps.append(R_work.copy())

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['$A$', 'After $H_1$', 'After $H_2 H_1$', 'After $H_3 H_2 H_1 = R$']
for ax, mat, title in zip(axes, R_steps, titles):
    im = ax.imshow(np.abs(mat), cmap='Blues', aspect='auto')
    # Annotate entries
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat[i, j]
            color = 'white' if abs(val) > 0.5 * np.max(np.abs(mat)) else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8, color=color)
    ax.set_title(title, fontsize=11)
    ax.set_xticks(range(mat.shape[1]))
    ax.set_yticks(range(mat.shape[0]))

plt.suptitle('Householder QR: progressive elimination', fontsize=13)
plt.tight_layout()
plt.show()

## Properties Verification

In [ ]:
# Verify all key properties of Householder reflectors
np.random.seed(7)
v = np.random.randn(5)
v = v / np.linalg.norm(v)
H = np.eye(5) - 2 * np.outer(v, v)

print("Householder reflector properties:")
print(f"  Orthogonal (H^TH = I): {np.allclose(H.T @ H, np.eye(5))}")
print(f"  Symmetric (H^T = H):   {np.allclose(H.T, H)}")
print(f"  Involutory (H^2 = I):  {np.allclose(H @ H, np.eye(5))}")
print(f"  det(H) = {np.linalg.det(H):.1f}  (should be -1)")
print(f"  Eigenvalues: {np.sort(np.linalg.eigvalsh(H)).round(10)}")
print(f"  (should be -1 once and +1 four times)")